# 04 -- Model selection: which interaction topology fits best?

We compare four chromosome interaction topologies:

| Label | xy partner(s) | xx term? |
|---|---|---|
| **poles** | both poles (2 partners) | no |
| **center** | pole midpoint (1 partner) | no |
| **poles\_and\_chroms** | both poles | yes |
| **center\_and\_chroms** | pole midpoint | yes |

Primary selection criterion: leave-one-cell-out one-step velocity MSE with
paired fold-difference uncertainty.
Secondary checks: rollout validation, bootstrap CIs, kernel physics checks.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from chromlearn import find_repo_root

ROOT = find_repo_root(Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd())

from chromlearn.io.catalog import load_condition
from chromlearn.io.trajectory import (
    TrimmedCell,
    get_partners,
    spindle_frame,
    trim_trajectory,
)
from chromlearn.model_fitting import FitConfig
from chromlearn.model_fitting.basis import BSplineBasis
from chromlearn.model_fitting.fit import (
    BootstrapResult,
    CVResult,
    RolloutCVResult,
    bootstrap_kernels,
    cross_validate,
    fit_model,
    paired_cv_differences,
    rollout_cross_validate,
)
from chromlearn.model_fitting.model import FittedModel
from chromlearn.model_fitting.plotting import plot_cv_curve, plot_kernels
from chromlearn.model_fitting.simulate import simulate_cell, simulate_trajectories

plt.rcParams["figure.dpi"] = 110

In [ ]:
cells_raw = load_condition("rpe18_ctr")
cells = [trim_trajectory(c, method="neb_ao_frac") for c in cells_raw]
print(f"Loaded {len(cells)} rpe18_ctr cells (trimmed to neb_ao_frac=0.5 window)")
for c in cells:
    T, _, N = c.chromosomes.shape
    print(f"  {c.cell_id}: {T} frames, {N} chromosomes")

## Basis domains

Basis support is fixed a priori from imaging resolution and spindle geometry,
not estimated from the data.  This avoids leaking held-out cell information
into the basis used during cross-validation.

- `r_min = 0.3 um`: below kinetochore tracking resolution.
- `r_max = 15.0 um`: conservative upper bound from RPE1 spindle geometry
  (pole-to-pole ~10-14 um; 15 um covers all plausible pairwise distances).

We use a single domain for both xx and xy interactions across all topologies.
Empirical distance distributions are plotted below as a sanity check.

In [ ]:
TOPOLOGIES = ["poles", "center", "poles_and_chroms", "center_and_chroms"]

R_MIN = 0.3   # um — tracking resolution floor
R_MAX = 15.0  # um — conservative spindle-scale upper bound

r_min_xx = R_MIN
r_max_xx = R_MAX
r_min_xy = R_MIN
r_max_xy = R_MAX

# For backward compat with code that indexes xy_domains[topology]
xy_domains: dict[str, tuple[float, float]] = {t: (R_MIN, R_MAX) for t in TOPOLOGIES}

print(f"Fixed basis domains: r_min={R_MIN}, r_max={R_MAX} um (all interactions, all topologies)")

In [ ]:
# Empirical distance distributions (sanity check that fixed domains cover the data)
xy_dists_by_topology: dict[str, list[float]] = {t: [] for t in TOPOLOGIES}
xx_dists_all: list[float] = []

for cell in cells:
    T, _, N = cell.chromosomes.shape
    chroms = cell.chromosomes  # (T, 3, N)

    for t in range(T):
        pos_t = chroms[t].T  # (N, 3)
        for i in range(N):
            if np.any(np.isnan(pos_t[i])):
                continue
            for j in range(i + 1, N):
                if np.any(np.isnan(pos_t[j])):
                    continue
                d = float(np.linalg.norm(pos_t[j] - pos_t[i]))
                if d > 1e-12:
                    xx_dists_all.append(d)

    for topology in TOPOLOGIES:
        partners = get_partners(cell, topology)  # (n_p, T, 3)
        for t in range(T):
            pos_t = chroms[t].T  # (N, 3)
            for p_idx in range(partners.shape[0]):
                partner_pos = partners[p_idx, t]  # (3,)
                for i in range(N):
                    if np.any(np.isnan(pos_t[i])):
                        continue
                    d = float(np.linalg.norm(partner_pos - pos_t[i]))
                    if d > 1e-12:
                        xy_dists_by_topology[topology].append(d)

xx_dists_all = np.array(xx_dists_all)
print(f"Collected {len(xx_dists_all):,} xx distance samples  "
      f"(observed range: {np.min(xx_dists_all):.2f} – {np.max(xx_dists_all):.2f} um)")
for t in TOPOLOGIES:
    arr = np.array(xy_dists_by_topology[t])
    print(f"  xy ({t}): {len(arr):,} samples  "
          f"(observed range: {np.min(arr):.2f} – {np.max(arr):.2f} um)")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].hist(xx_dists_all, bins=60, color="C2", edgecolor="k", alpha=0.7)
axes[0].axvline(R_MIN, color="r", linestyle="--", linewidth=1.5, label=f"r_min={R_MIN}")
axes[0].axvline(R_MAX, color="r", linestyle="-", linewidth=1.5, label=f"r_max={R_MAX}")
axes[0].set_title("Chromosome-chromosome (xx)")
axes[0].set_xlabel("Distance (um)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=7)

for idx, topology in enumerate(TOPOLOGIES):
    ax = axes[idx + 1]
    ax.hist(xy_dists_by_topology[topology], bins=60, color=f"C{idx}", edgecolor="k", alpha=0.7)
    ax.axvline(R_MIN, color="r", linestyle="--", linewidth=1.5, label=f"r_min={R_MIN}")
    ax.axvline(R_MAX, color="r", linestyle="-", linewidth=1.5, label=f"r_max={R_MAX}")
    ax.set_title(f"xy ({topology})")
    ax.set_xlabel("Distance (um)")
    ax.legend(fontsize=7)

fig.suptitle("Observed distance distributions — fixed basis domains marked in red")
fig.tight_layout()
plt.show()

## Fit all four models

We use:
- `n_basis = 10` B-spline basis functions for both xx and xy kernels
- `lambda_ridge = lambda_rough = 1e-3`
- `basis_eval_mode = "ito"` (current positions, standard SFI)

Domain parameters are fixed a priori (see basis-domain section above).

In [ ]:
N_BASIS = 10
LAMBDA = 1e-3

configs: dict[str, FitConfig] = {}
for topology in TOPOLOGIES:
    configs[topology] = FitConfig(
        topology=topology,
        n_basis_xx=N_BASIS,
        n_basis_xy=N_BASIS,
        r_min_xx=R_MIN,
        r_max_xx=R_MAX,
        r_min_xy=R_MIN,
        r_max_xy=R_MAX,
        basis_type="bspline",
        lambda_ridge=LAMBDA,
        lambda_rough=LAMBDA,
        basis_eval_mode="ito",
        dt=5.0,
    )

print("Fitting models...")
models: dict[str, FittedModel] = {}
for topology in TOPOLOGIES:
    models[topology] = fit_model(cells, configs[topology])
    m = models[topology]
    print(f"  {topology:<22}  D_x={m.D_x:.8f} um^2/s  "
          f"n_params={m.theta.size}")

## Cross-validation comparison

Leave-one-cell-out CV: fit on 12 cells, evaluate mean squared error on the
held-out cell.  Lower is better.

**Primary model-selection criterion**: leave-one-cell-out one-step velocity
MSE, averaged equally across held-out cells.  Paired fold-by-fold loss
differences and their SEs are reported to assess whether score gaps between
topologies are meaningful.

In [ ]:
print("Running leave-one-cell-out cross-validation (4 topologies × 13 folds)...")
cv_results: dict[str, CVResult] = {}
for topology in TOPOLOGIES:
    cv_results[topology] = cross_validate(cells, configs[topology])
    r = cv_results[topology]
    print(f"  {topology:<22}  MSE={r.mean_error:.8f}  (fold SD={r.fold_sd:.8f}, SE={r.fold_se:.8f})")

In [ ]:
fig = plot_cv_curve(cv_results)
fig.suptitle("Leave-one-cell-out CV — mean squared velocity prediction error", y=1.02)
plt.show()

# Bar chart with per-cell breakdown
n_topo = len(TOPOLOGIES)
n_cells = len(cells)
x = np.arange(n_cells)
width = 0.8 / n_topo

fig, ax = plt.subplots(figsize=(12, 4.5))
for idx, topology in enumerate(TOPOLOGIES):
    offset = (idx - n_topo / 2 + 0.5) * width
    ax.bar(x + offset, cv_results[topology].held_out_errors, width,
           label=topology, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([c.cell_id.split("_")[-1] for c in cells], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("MSE (um/s)^2")
ax.set_title("Per-cell CV errors — all topologies")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

# Summary printout
print("\nCV summary (sorted by mean MSE):")
sorted_topo_cv = sorted(TOPOLOGIES, key=lambda t: cv_results[t].mean_error)
best_cv = cv_results[sorted_topo_cv[0]].mean_error
for rank, topology in enumerate(sorted_topo_cv):
    r = cv_results[topology]
    delta = r.mean_error - best_cv
    print(f"  #{rank + 1}  {topology:<22}  MSE={r.mean_error:.8f}  "
          f"(Δbest={delta:+.2e}, SE={r.fold_se:.2e})")

## Bootstrap kernel confidence bands

250 cell-level resamples per topology; shaded band = 5–95% quantile interval.

In [ ]:
N_BOOT = 50
boot_rng = np.random.default_rng(42)

print(f"Bootstrapping kernels ({N_BOOT} resamples × 4 topologies)...")
boot_results: dict[str, BootstrapResult] = {}
for topology in TOPOLOGIES:
    boot_results[topology] = bootstrap_kernels(
        cells, configs[topology], n_boot=N_BOOT, rng=boot_rng
    )
    print(f"  {topology} done.")

In [ ]:
# Plot each topology's kernels with bootstrap bands
for topology in TOPOLOGIES:
    fig = plot_kernels(models[topology], bootstrap=boot_results[topology])
    fig.suptitle(f"Learned kernels — topology: {topology}", y=1.02)
    plt.show()

## Physical plausibility of the chromosome-chromosome kernel

For topologies that include a chromosome-chromosome term (`poles_and_chroms`
and `center_and_chroms`), we inspect whether the learned kernel is
physically sensible:

- Sign convention: forces are assembled as `f(r) * (x_j - x_i) / r`, so
  positive values are attractive and negative values are repulsive.
- **Expected**: a repulsive barrier at short distances (excluded volume,
  r ≲ 1 um), i.e. negative `f_xx(r)` at small `r`.
- **Red flag**: short-range attraction (positive force at small `r`), which
  would pull chromosomes into one another.

We flag any model where f_xx at r < 1.5 um becomes positive.

In [ ]:
chroms_topologies = [t for t in TOPOLOGIES if t in ("poles_and_chroms", "center_and_chroms")]
r_probe = np.linspace(r_min_xx, r_max_xx, 400)
SHORT_R_THRESHOLD = 1.5  # um — below this we expect repulsion, not attraction

fig, axes = plt.subplots(1, len(chroms_topologies), figsize=(7 * len(chroms_topologies), 4.5),
                          squeeze=False)

for col, topology in enumerate(chroms_topologies):
    ax = axes[0, col]
    m = models[topology]
    boot = boot_results[topology]

    f_vals = m.evaluate_kernel("xx", r_probe)

    # Bootstrap band
    phi = m.basis_xx.evaluate(r_probe)
    theta_xx_samples = boot.theta_samples[:, : m.n_basis_xx]
    curves = phi @ theta_xx_samples.T
    lo = np.percentile(curves, 5, axis=1)
    hi = np.percentile(curves, 95, axis=1)
    ax.fill_between(r_probe, lo, hi, color="C0", alpha=0.2, label="5–95% CI")

    ax.plot(r_probe, f_vals, "C0", linewidth=2, label="Mean fit")
    ax.axhline(0.0, color="0.5", linestyle="--", linewidth=0.8)
    ax.axvline(SHORT_R_THRESHOLD, color="orange", linestyle=":", linewidth=1.2,
               label=f"r={SHORT_R_THRESHOLD} um")
    ax.set_xlabel("Distance (um)")
    ax.set_ylabel("Force")
    ax.set_title(f"f_xx — {topology}")
    ax.legend(fontsize=8)

    # Diagnosis
    short_r_mask = r_probe < SHORT_R_THRESHOLD
    max_short_r = float(np.max(f_vals[short_r_mask])) if short_r_mask.any() else np.nan
    if max_short_r > 0:
        print(f"  [WARNING] {topology}: f_xx is ATTRACTIVE at short range "
              f"(max={max_short_r:.4f} at r < {SHORT_R_THRESHOLD} um). "
              "Likely an artifact — excluded-volume physics expects repulsion here.")
    else:
        print(f"  [OK] {topology}: f_xx is repulsive or neutral at short range "
              f"(max={max_short_r:.4f} at r < {SHORT_R_THRESHOLD} um).")

fig.suptitle("Chromosome-chromosome kernel f_xx — physical plausibility check")
fig.tight_layout()
plt.show()

# Also plot f_xy for chroms topologies alongside poles/center for comparison
fig, axes = plt.subplots(1, len(TOPOLOGIES), figsize=(6 * len(TOPOLOGIES), 4.5), squeeze=False)
r_xy_probe = {t: np.linspace(R_MIN, R_MAX, 400) for t in TOPOLOGIES}

for col, topology in enumerate(TOPOLOGIES):
    ax = axes[0, col]
    m = models[topology]
    boot = boot_results[topology]
    r_p = r_xy_probe[topology]

    f_vals = m.evaluate_kernel("xy", r_p)
    phi = m.basis_xy.evaluate(r_p)
    theta_xy_samples = boot.theta_samples[:, m.n_basis_xx:]
    curves = phi @ theta_xy_samples.T
    lo = np.percentile(curves, 5, axis=1)
    hi = np.percentile(curves, 95, axis=1)

    ax.fill_between(r_p, lo, hi, color=f"C{col}", alpha=0.2, label="5–95% CI")
    ax.plot(r_p, f_vals, f"C{col}", linewidth=2, label="Mean fit")
    ax.axhline(0.0, color="0.5", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Distance (um)")
    ax.set_ylabel("Force")
    ax.set_title(f"f_xy — {topology}")
    ax.legend(fontsize=8)

fig.suptitle("Chromosome-partner kernel f_xy — all topologies")
fig.tight_layout()
plt.show()

## Rollout validation: qualitative cells and aggregate holdout scoring

One-step CV is useful for local velocity prediction, but it does not tell us
whether a fitted model generates plausible chromosome trajectories when run
forward.  We therefore look at rollout validation in two complementary ways:

1. A few representative cells with **one simulated rollout per model** to
   inspect whether the simulated trajectories qualitatively resemble the real
   spindle-frame traces.
2. **Leave-one-cell-out rollout validation** that aggregates across all cells
   and scores axial/radial summary trajectories and final-frame
   distributions.

In [ ]:
EXAMPLE_CELL_IDX = 1
example_cell = cells[EXAMPLE_CELL_IDX]
T, _, N_chrom = example_cell.chromosomes.shape
n_steps = T - 1
x0 = example_cell.chromosomes[0].T
sf_real = spindle_frame(example_cell)

QUAL_CELL_IDXS = sorted({0, len(cells) // 2, len(cells) - 1})
QUAL_N_TRACES = 6
ROLLOUT_REPS = 4
ROLLOUT_HORIZONS = (1, 5, 10, 20)


def _simulate_cell_once(cell: TrimmedCell, model: FittedModel, seed: int):
    traj, sim_cell = simulate_cell(cell, model, rng=np.random.default_rng(seed))
    return traj, spindle_frame(sim_cell)


def _representative_trace_indices(sf_data, n_traces: int) -> np.ndarray:
    valid = np.flatnonzero(
        np.all(np.isfinite(sf_data.axial), axis=0)
        & np.all(np.isfinite(sf_data.radial), axis=0)
    )
    if valid.size == 0:
        return valid
    if valid.size <= n_traces:
        return valid

    # Spread displayed traces across the initial radial distribution so the
    # thin-line examples are less biased than "first N valid chromosomes".
    radial0 = sf_data.radial[0, valid]
    order = valid[np.argsort(radial0)]
    positions = np.linspace(0, order.size - 1, n_traces, dtype=int)
    return order[positions]


def _interp_to_unit_grid(values: np.ndarray, n_grid: int = 100) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return np.full(n_grid, np.nan, dtype=np.float64)
    if values.size == 1:
        return np.full(n_grid, values[0], dtype=np.float64)
    src = np.linspace(0.0, 1.0, values.size)
    dst = np.linspace(0.0, 1.0, n_grid)
    return np.interp(dst, src, values)


print("Qualitative rollout check on representative cells (1 rollout per model)...")
for cell_idx in QUAL_CELL_IDXS:
    cell = cells[cell_idx]
    sf_cell_real = spindle_frame(cell)
    time_axis = np.arange(cell.chromosomes.shape[0]) * cell.dt
    trace_idx = _representative_trace_indices(sf_cell_real, QUAL_N_TRACES)

    fig, axes = plt.subplots(2, len(TOPOLOGIES), figsize=(5 * len(TOPOLOGIES), 7), squeeze=False)
    print(f"  {cell.cell_id}: {cell.chromosomes.shape[0]} frames, {cell.chromosomes.shape[2]} chromosomes")

    for col, topology in enumerate(TOPOLOGIES):
        sim_seed = 1000 + 100 * cell_idx + col
        _traj, sf_sim = _simulate_cell_once(cell, models[topology], seed=sim_seed)

        ax_axial = axes[0, col]
        ax_radial = axes[1, col]
        color = f"C{col}"

        for chrom_idx in trace_idx:
            ax_axial.plot(time_axis, sf_cell_real.axial[:, chrom_idx], color="k", alpha=0.15, linewidth=0.8)
            ax_axial.plot(time_axis, sf_sim.axial[:, chrom_idx], color=color, alpha=0.18, linewidth=0.8)
            ax_radial.plot(time_axis, sf_cell_real.radial[:, chrom_idx], color="k", alpha=0.15, linewidth=0.8)
            ax_radial.plot(time_axis, sf_sim.radial[:, chrom_idx], color=color, alpha=0.18, linewidth=0.8)

        real_axial_subset_mean = np.nanmean(sf_cell_real.axial[:, trace_idx], axis=1)
        sim_axial_subset_mean = np.nanmean(sf_sim.axial[:, trace_idx], axis=1)
        real_radial_subset_mean = np.nanmean(sf_cell_real.radial[:, trace_idx], axis=1)
        sim_radial_subset_mean = np.nanmean(sf_sim.radial[:, trace_idx], axis=1)

        ax_axial.plot(time_axis, real_axial_subset_mean, "k-", linewidth=2.0, label="Displayed real mean")
        ax_axial.plot(time_axis, sim_axial_subset_mean, color=color, linestyle="--", linewidth=2.0,
                      label="Displayed rollout mean")
        ax_radial.plot(time_axis, real_radial_subset_mean, "k-", linewidth=2.0, label="Displayed real mean")
        ax_radial.plot(time_axis, sim_radial_subset_mean, color=color, linestyle="--", linewidth=2.0,
                       label="Displayed rollout mean")

        ax_axial.set_title(f"Axial — {topology}", fontsize=9)
        ax_radial.set_title(f"Radial — {topology}", fontsize=9)
        ax_axial.set_xlabel("Time (s)")
        ax_radial.set_xlabel("Time (s)")
        if col == 0:
            ax_axial.set_ylabel("Axial position (um)")
            ax_radial.set_ylabel("Radial distance (um)")
        ax_axial.legend(fontsize=7)
        ax_radial.legend(fontsize=7)

    fig.suptitle(f"Single-rollout qualitative check — {cell.cell_id}\n"
                 "Thin lines = representative chromosome traces, thick lines = means of displayed traces")
    fig.tight_layout()
    plt.show()

In [ ]:
AGG_NORM_GRID = 100
print("Aggregating full-data forward simulations across all cells...")

agg_real_axial = []
agg_real_radial = []
agg_sim_axial: dict[str, list[np.ndarray]] = {t: [] for t in TOPOLOGIES}
agg_sim_radial: dict[str, list[np.ndarray]] = {t: [] for t in TOPOLOGIES}

for cell_idx, cell in enumerate(cells):
    sf_cell_real = spindle_frame(cell)
    agg_real_axial.append(_interp_to_unit_grid(np.nanmean(sf_cell_real.axial, axis=1), n_grid=AGG_NORM_GRID))
    agg_real_radial.append(_interp_to_unit_grid(np.nanmean(sf_cell_real.radial, axis=1), n_grid=AGG_NORM_GRID))

    for topo_index, topology in enumerate(TOPOLOGIES):
        _traj, sf_sim = _simulate_cell_once(
            cell,
            models[topology],
            seed=10_000 + 100 * cell_idx + topo_index,
        )
        agg_sim_axial[topology].append(
            _interp_to_unit_grid(np.nanmean(sf_sim.axial, axis=1), n_grid=AGG_NORM_GRID)
        )
        agg_sim_radial[topology].append(
            _interp_to_unit_grid(np.nanmean(sf_sim.radial, axis=1), n_grid=AGG_NORM_GRID)
        )

agg_real_axial = np.stack(agg_real_axial, axis=0)
agg_real_radial = np.stack(agg_real_radial, axis=0)
norm_time = np.linspace(0.0, 1.0, AGG_NORM_GRID)

fig, axes = plt.subplots(2, len(TOPOLOGIES), figsize=(5 * len(TOPOLOGIES), 7), squeeze=False)
for col, topology in enumerate(TOPOLOGIES):
    ax_axial = axes[0, col]
    ax_radial = axes[1, col]

    sim_axial_stack = np.stack(agg_sim_axial[topology], axis=0)
    sim_radial_stack = np.stack(agg_sim_radial[topology], axis=0)

    real_axial_mean = np.nanmean(agg_real_axial, axis=0)
    real_axial_std = np.nanstd(agg_real_axial, axis=0)
    real_radial_mean = np.nanmean(agg_real_radial, axis=0)
    real_radial_std = np.nanstd(agg_real_radial, axis=0)

    sim_axial_mean = np.nanmean(sim_axial_stack, axis=0)
    sim_axial_std = np.nanstd(sim_axial_stack, axis=0)
    sim_radial_mean = np.nanmean(sim_radial_stack, axis=0)
    sim_radial_std = np.nanstd(sim_radial_stack, axis=0)

    color = f"C{col}"
    ax_axial.plot(norm_time, real_axial_mean, "k-", linewidth=2.0, label="Real mean")
    ax_axial.fill_between(
        norm_time,
        real_axial_mean - real_axial_std,
        real_axial_mean + real_axial_std,
        color="k",
        alpha=0.12,
    )
    ax_axial.plot(norm_time, sim_axial_mean, color=color, linestyle="--", linewidth=2.0, label="Sim mean")
    ax_axial.fill_between(
        norm_time,
        sim_axial_mean - sim_axial_std,
        sim_axial_mean + sim_axial_std,
        color=color,
        alpha=0.18,
    )

    ax_radial.plot(norm_time, real_radial_mean, "k-", linewidth=2.0, label="Real mean")
    ax_radial.fill_between(
        norm_time,
        real_radial_mean - real_radial_std,
        real_radial_mean + real_radial_std,
        color="k",
        alpha=0.12,
    )
    ax_radial.plot(norm_time, sim_radial_mean, color=color, linestyle="--", linewidth=2.0, label="Sim mean")
    ax_radial.fill_between(
        norm_time,
        sim_radial_mean - sim_radial_std,
        sim_radial_mean + sim_radial_std,
        color=color,
        alpha=0.18,
    )

    ax_axial.set_title(f"Axial — {topology}", fontsize=9)
    ax_radial.set_title(f"Radial — {topology}", fontsize=9)
    ax_axial.set_xlabel("Normalized progress through trimmed window")
    ax_radial.set_xlabel("Normalized progress through trimmed window")
    if col == 0:
        ax_axial.set_ylabel("Mean axial position (um)")
        ax_radial.set_ylabel("Mean radial distance (um)")
    ax_axial.legend(fontsize=7)
    ax_radial.legend(fontsize=7)

fig.suptitle("Aggregate forward simulation across all cells\n"
             "Real initial conditions and pole trajectories, normalized in time")
fig.tight_layout()
plt.show()

In [ ]:
print("Running leave-one-cell-out rollout validation "
      f"({len(TOPOLOGIES)} topologies × {len(cells)} folds × {ROLLOUT_REPS} rollouts)...")
rollout_results: dict[str, RolloutCVResult] = {}
for topo_index, topology in enumerate(TOPOLOGIES):
    rollout_results[topology] = rollout_cross_validate(
        cells,
        configs[topology],
        n_reps=ROLLOUT_REPS,
        horizons=ROLLOUT_HORIZONS,
        rng=np.random.default_rng(200 + topo_index),
    )
    rr = rollout_results[topology]
    print(f"  {topology:<22}  axial_MSE={np.nanmean(rr.axial_mse):.5f}  "
          f"radial_MSE={np.nanmean(rr.radial_mse):.5f}  "
          f"endpoint_MSE={np.nanmean(rr.endpoint_mean_error):.5f}  "
          f"final_W1(ax,rad)=({np.nanmean(rr.final_axial_wasserstein):.4f}, "
          f"{np.nanmean(rr.final_radial_wasserstein):.4f})")

In [ ]:
rollout_time_score = {
    topology: float(np.nanmean(rollout_results[topology].axial_mse)
                    + np.nanmean(rollout_results[topology].radial_mse))
    for topology in TOPOLOGIES
}
rollout_endpoint_score = {
    topology: float(np.nanmean(rollout_results[topology].endpoint_mean_error))
    for topology in TOPOLOGIES
}
rollout_dist_score = {
    topology: float(np.nanmean(rollout_results[topology].final_axial_wasserstein)
                    + np.nanmean(rollout_results[topology].final_radial_wasserstein))
    for topology in TOPOLOGIES
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
x = np.arange(len(TOPOLOGIES))

axes[0].bar(x, [rollout_time_score[t] for t in TOPOLOGIES], color=[f"C{i}" for i in range(len(TOPOLOGIES))])
axes[0].set_xticks(x)
axes[0].set_xticklabels(TOPOLOGIES, rotation=45, ha="right")
axes[0].set_ylabel("Mean rollout trajectory error (um^2)")
axes[0].set_title("LOOCV rollout score\n(axial MSE + radial MSE)")

axes[1].bar(x, [rollout_endpoint_score[t] for t in TOPOLOGIES], color=[f"C{i}" for i in range(len(TOPOLOGIES))])
axes[1].set_xticks(x)
axes[1].set_xticklabels(TOPOLOGIES, rotation=45, ha="right")
axes[1].set_ylabel("Endpoint mean error (um^2)")
axes[1].set_title("LOOCV endpoint-only score\n(final axial/radial mean mismatch)")

axes[2].bar(x, [rollout_dist_score[t] for t in TOPOLOGIES], color=[f"C{i}" for i in range(len(TOPOLOGIES))])
axes[2].set_xticks(x)
axes[2].set_xticklabels(TOPOLOGIES, rotation=45, ha="right")
axes[2].set_ylabel("Final-frame W1 distance (um)")
axes[2].set_title("LOOCV final distribution mismatch\n(axial W1 + radial W1)")

for topo_index, topology in enumerate(TOPOLOGIES):
    axes[3].plot(
        rollout_results[topology].horizons,
        np.nanmean(rollout_results[topology].horizon_errors, axis=0),
        marker="o",
        linewidth=1.8,
        label=topology,
    )
axes[3].set_xlabel("Forecast horizon (frames)")
axes[3].set_ylabel("Combined axial/radial error (um^2)")
axes[3].set_title("Held-out forecast error vs horizon")
axes[3].legend(fontsize=8)

fig.suptitle("Aggregate rollout validation across held-out cells")
fig.tight_layout()
plt.show()

In [ ]:
# Detailed LOOCV rollout table: per-metric means ± SE across held-out cells
print("\nLOOCV rollout validation — detailed summary (mean ± SE across held-out cells)")
print("=" * 120)
print(f"{'Topology':<22} {'Axial MSE':>14} {'Radial MSE':>14} {'Endpoint':>14} "
      f"{'W1 axial':>14} {'W1 radial':>14}")
print("-" * 120)

n_cv_cells = len(cells)
for topology in TOPOLOGIES:
    rr = rollout_results[topology]
    def _fmt(arr):
        m = np.nanmean(arr)
        se = np.nanstd(arr) / np.sqrt(np.sum(np.isfinite(arr)))
        return f"{m:.4f} ± {se:.4f}"
    print(f"  {topology:<22} {_fmt(rr.axial_mse):>14} {_fmt(rr.radial_mse):>14} "
          f"{_fmt(rr.endpoint_mean_error):>14} "
          f"{_fmt(rr.final_axial_wasserstein):>14} {_fmt(rr.final_radial_wasserstein):>14}")

print("=" * 120)

# Horizon-resolved table
print("\nHeld-out forecast error by horizon (mean ± SE, axial+radial combined)")
horizons = rollout_results[TOPOLOGIES[0]].horizons
header = f"{'Topology':<22}" + "".join(f"  h={h:<5}" for h in horizons)
print(header)
for topology in TOPOLOGIES:
    rr = rollout_results[topology]
    vals = []
    for hi in range(len(horizons)):
        col = rr.horizon_errors[:, hi]
        m = np.nanmean(col)
        se = np.nanstd(col) / np.sqrt(np.sum(np.isfinite(col)))
        vals.append(f"{m:.4f}±{se:.4f}")
    print(f"  {topology:<22}" + "".join(f"  {v:<7}" for v in vals))

## Model selection summary

**Primary criterion**: leave-one-cell-out one-step velocity MSE, with paired
fold-difference SEs to assess whether score gaps are meaningful.

**Secondary criterion**: LOOCV rollout validation (pathwise MSE, endpoint
error, final-frame distributional metrics).  This is the strongest test of
whether the learned dynamics reproduce real trajectories, but carries more
Monte Carlo noise than 1-step prediction.

**Qualitative checks**: full-data forward simulations and kernel-shape /
physics plausibility (above).

In [ ]:
sorted_topo = sorted(TOPOLOGIES, key=lambda t: cv_results[t].mean_error)
best_mean_topo = sorted_topo[0]
paired = paired_cv_differences(cv_results, reference=best_mean_topo)

print("Primary criterion: leave-one-cell-out 1-step velocity MSE")
print("=" * 105)
print(f"{'Topology':<22} {'CV MSE':>12} {'SE':>10} {'Δ vs best':>10} {'SE(Δ)':>10} {'Δ/SE(Δ)':>8} "
      f"{'D_x':>12} {'n_params':>9}")
print("-" * 105)

for topology in sorted_topo:
    r = cv_results[topology]
    m = models[topology]
    mean_diff, se_diff = paired[topology]
    ratio = mean_diff / se_diff if se_diff > 0 and se_diff < np.inf else 0.0
    print(f"  {topology:<22} {r.mean_error:>12.8f} {r.fold_se:>10.2e} "
          f"{mean_diff:>+10.2e} {se_diff:>10.2e} {ratio:>8.2f} "
          f"{m.D_x:>12.8f} {m.theta.size:>9}")

print("=" * 105)
print(f"  Best mean: {best_mean_topo}")
print(f"  Note: Δ/SE(Δ) < 1 means the gap is within one paired SE of zero.")

print("\nSecondary criterion: LOOCV rollout validation (means across held-out cells)")
print(f"  {'Topology':<22} {'Axial+Rad MSE':>14} {'Endpoint':>10} {'W1 total':>10}")
for t in sorted(TOPOLOGIES, key=lambda t: rollout_time_score[t]):
    print(f"  {t:<22} {rollout_time_score[t]:>14.4f} "
          f"{rollout_endpoint_score[t]:>10.4f} {rollout_dist_score[t]:>10.4f}")

best_topology = best_mean_topo

# Final kernel plot for the winner
print(f"\nFinal kernel plot for best model ({best_topology}):")
fig = plot_kernels(models[best_topology], bootstrap=boot_results[best_topology])
fig.suptitle(f"Best model kernels — {best_topology}", y=1.02)
plt.show()

## Note on diffusion

Spatially-varying diffusion D(x) analysis has been moved to
**Notebook 06 (06_diffusion_landscape.py)**, which provides a thorough
investigation with multiple estimators, per-cell consistency checks, and
comparison across coordinate axes.